# Build From Scratch — Author a Semantic View, Agent, and Evals

The [quickstart notebook](agent_management_quickstart.ipynb) shows how to **operate** the framework
(which `agent-mgmt-*` command does what). This notebook shows how to **author** the artifacts the
framework manages — starting from nothing and ending with a working, evaluated agent.

You will build a small, self-contained example end to end, using real data that already exists in your
Snowflake database:

```mermaid
flowchart LR
    tbl[demo table] --> sv[semantic view]
    sv --> agent[agent]
    agent --> smoke[smoke test]
    smoke --> ver[versioning + promotion]
    ver --> golden[golden dataset + eval]
    golden --> repo[promote into repo]
```

Each step uses the **same library APIs and primitives** the production CI/CD pipeline uses, so what you
build here transfers directly into the repo (covered in the final section).

## What you'll create

| Artifact | Built with | Demo object |
|---|---|---|
| Demo table | `CREATE TABLE AS SELECT` from existing marts | `SANDBOX.DEMO_DAILY_INCIDENTS` |
| Semantic view | `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML` | `SANDBOX.SEM_DEMO_INCIDENTS` |
| Agent | `deploy_agent()` (versioning path) | `AGENTS.INCIDENT_SUMMARY_DEMO` |
| Golden dataset + eval | YAML authoring + `agent-mgmt-eval-agent` | 3 questions |

A teardown cell at the end drops everything, so the demo leaves no residue.

## Prerequisites

This notebook is **live** — it creates and drops real Snowflake objects (in an isolated `SANDBOX`
schema). You need:

1. The package installed: `pip install -e ..` (run the install cell once).
2. Snowflake credentials available to the connector, e.g. a named connection:

   ```bash
   export SNOWFLAKE_CONNECTION_NAME=myconnection
   ```

   or explicit key-pair auth (`SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PRIVATE_KEY_PATH`).
3. A database with the ski-resort `MARTS` tables (the demo reads `MARTS.FACT_INCIDENTS` and
   `MARTS.DIM_DATE`). This is the `dev` database by default.

Set `RUN_LIVE = False` to read through the notebook without touching Snowflake (object-creating cells
become no-ops and just print what they *would* do).

In [ ]:
# Install the package once (uncomment in a fresh environment).
# %pip install -e ..

import os
from pathlib import Path

import agent_management

print("agent_management version:", agent_management.__version__)

## Parameters & connection

- `ENV` — which environment's database/role/warehouse to use (`dev` recommended for this demo).
- `SANDBOX_SCHEMA` — isolated schema for all demo objects; dropped at the end.
- `RUN_LIVE` — `True` actually creates objects; `False` is a dry read-through.

The library resolves all connection + naming settings from `project.yml` + `environments/<env>.env.yml`
via `load_env_config()`, and opens a connection with `connect()` — the same helper every CLI uses.

In [ ]:
from agent_management.utils.config import (
    get_agents_schema,
    get_database,
    load_env_config,
    load_project_config,
)
from agent_management.utils.snowflake_client import connect

# ---- Parameters -------------------------------------------------------------
ENV = "dev"               # dev or prod
SANDBOX_SCHEMA = "SANDBOX"  # isolated schema for demo objects (created + dropped)
RUN_LIVE = True           # False = read-through, no Snowflake objects created
# The demo CREATEs a schema, table, semantic view, and agent, so it needs a role
# with CREATE privileges. The env deploy role (e.g. AM_DEPLOY_ROLE_DEV) usually
# cannot create schemas — ACCOUNTADMIN is the simplest role that can run the
# whole demo. Change this if you have a dedicated sandbox role.
DEMO_ROLE = "ACCOUNTADMIN"
# -----------------------------------------------------------------------------

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("AGENT_MGMT_PROJECT_CONFIG", str(REPO_ROOT / "project.yml"))

project = load_project_config()
env_cfg = load_env_config(env=ENV)

DATABASE = get_database(env_cfg)                 # e.g. AM_SKI_RESORT_DEV
AGENTS_SCHEMA = get_agents_schema(env_cfg)       # e.g. AM_SKI_RESORT_DEV.AGENTS
SANDBOX_FQN = f"{DATABASE}.{SANDBOX_SCHEMA}"

print(f"ENV={ENV}  RUN_LIVE={RUN_LIVE}")
print(f"Database      : {DATABASE}")
print(f"Sandbox schema: {SANDBOX_FQN}")
print(f"Agents schema : {AGENTS_SCHEMA}")

# Open a connection (None when not live so the notebook still reads cleanly).
# ---- Connection ----------------------------------------------------------
# Auth resolution order:
#   1. CONNECTION_NAME / SNOWFLAKE_USER / PRIVATE_KEY_PATH set here or via env
#   2. auto-discover a connections.toml entry whose account matches this env
# Auto-discovery means you usually don't need to set anything: it finds the
# connection for the target account (and ignores your global default_connection
# if that points at a different account).
CONNECTION_NAME = os.environ.get("SNOWFLAKE_CONNECTION_NAME") or ""   # e.g. "myconnection"
SNOWFLAKE_USER = os.environ.get("SNOWFLAKE_USER") or ""              # e.g. "JDEMLOW"
PRIVATE_KEY_PATH = os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH") or ""  # e.g. "~/.snowflake/keys/key.p8"

TARGET_ACCOUNT = str(env_cfg.get("snowflake", {}).get("account", ""))


def _read_connections_toml() -> dict:
    import tomllib

    path = Path.home() / ".snowflake" / "connections.toml"
    return tomllib.loads(path.read_text()) if path.exists() else {}


def _autodiscover_connection(account: str):
    """Find a connections.toml entry whose account matches `account`.

    Prefers non-externalbrowser (key-pair/JWT) connections for non-interactive
    runs. Returns (connection_name, user) or (None, None)."""
    data = _read_connections_toml()
    matches = [
        (name, entry)
        for name, entry in data.items()
        if isinstance(entry, dict)
        and str(entry.get("account", "")).lower() == account.lower()
    ]
    if not matches:
        return None, None
    matches.sort(key=lambda ne: ne[1].get("authenticator", "") == "externalbrowser")
    name, entry = matches[0]
    return name, (entry.get("user") or entry.get("username"))


# If nothing was set explicitly, auto-discover a connection for this account.
if not (CONNECTION_NAME or PRIVATE_KEY_PATH or SNOWFLAKE_USER):
    auto_name, auto_user = _autodiscover_connection(TARGET_ACCOUNT)
    if auto_name:
        CONNECTION_NAME = auto_name
        SNOWFLAKE_USER = SNOWFLAKE_USER or (auto_user or "")
        print(f"Auto-selected connection '{CONNECTION_NAME}' for account {TARGET_ACCOUNT}.")

# SnowflakeConfig requires `user` even with a named connection; resolve it from
# connections.toml when only a connection name is known.
overrides: dict = {}
if CONNECTION_NAME:
    overrides["connection_name"] = CONNECTION_NAME
    resolved_user = SNOWFLAKE_USER or _autodiscover_connection(TARGET_ACCOUNT)[1]
    if resolved_user:
        overrides["user"] = resolved_user
if SNOWFLAKE_USER:
    overrides["user"] = SNOWFLAKE_USER
if PRIVATE_KEY_PATH:
    overrides["private_key_path"] = PRIVATE_KEY_PATH

conn = None
if RUN_LIVE:
    try:
        conn = connect(env_cfg, **overrides)
        # Use a role + warehouse that can create the demo objects.
        cur = conn.cursor()
        cur.execute(f"USE ROLE {DEMO_ROLE}")
        cur.execute(f"USE WAREHOUSE {env_cfg['snowflake']['warehouse']}")
        cur.close()
        print(f"Connected to Snowflake (role={DEMO_ROLE}).")
    except Exception as exc:
        print(f"Connection failed: {exc}\n")
        print(f"No connection found for account {TARGET_ACCOUNT}. Set one explicitly above, e.g.:")
        print('    CONNECTION_NAME = "myconnection"')
        print("  or key-pair auth:")
        print('    SNOWFLAKE_USER = "JDEMLOW"; PRIVATE_KEY_PATH = "~/.snowflake/keys/key.p8"')
        raise


def run_sql(sql: str, fetch: bool = False):
    """Execute SQL on the shared connection; print a short preview."""
    preview = " ".join(sql.split())[:120]
    print(f"SQL> {preview}{'...' if len(preview) == 120 else ''}")
    if not RUN_LIVE:
        print("     [skipped — RUN_LIVE=False]")
        return None
    cur = conn.cursor()
    try:
        cur.execute(sql)
        rows = cur.fetchall() if fetch else None
        return rows
    finally:
        cur.close()

## Step 1 — Create a self-contained demo table

A semantic view needs a base table. To keep this example self-contained and meaningful, we build a small
aggregate table from data that already exists in your marts: a daily incident summary derived from
`MARTS.FACT_INCIDENTS` joined to `MARTS.DIM_DATE`.

This lands in the isolated `SANDBOX` schema and is dropped in the teardown step.

In [ ]:
# Create the sandbox schema and a small, real-data demo table.
run_sql(f"CREATE SCHEMA IF NOT EXISTS {SANDBOX_FQN}")

run_sql(f"""
CREATE OR REPLACE TABLE {SANDBOX_FQN}.DEMO_DAILY_INCIDENTS AS
SELECT
    d.FULL_DATE                              AS INCIDENT_DATE,
    d.SKI_SEASON                             AS SKI_SEASON,
    COUNT(*)                                 AS INCIDENT_COUNT,
    ROUND(AVG(i.SEVERITY_SCORE), 2)          AS AVG_SEVERITY,
    ROUND(AVG(i.PATROL_RESPONSE_MINUTES), 1) AS AVG_RESPONSE_MINUTES
FROM {DATABASE}.MARTS.FACT_INCIDENTS i
JOIN {DATABASE}.MARTS.DIM_DATE d
  ON i.INCIDENT_DATE = d.FULL_DATE
GROUP BY d.FULL_DATE, d.SKI_SEASON
""")

# Preview a few rows.
rows = run_sql(
    f"SELECT * FROM {SANDBOX_FQN}.DEMO_DAILY_INCIDENTS ORDER BY INCIDENT_DATE DESC LIMIT 5",
    fetch=True,
)
if rows:
    for r in rows:
        print(r)

## Step 2 — Author a semantic view

A semantic view is the data contract a Cortex Analyst tool queries. It declares:

- **tables** — the base table(s), with a fully-qualified `base_table`.
- **dimensions** — columns you filter/group by.
- **facts** — raw numeric columns.
- **metrics** — reusable aggregations (this is what makes text-to-SQL reliable).

We author the YAML as a string. Note the `{{ env.database }}` placeholder: the library's
`render_string()` substitutes it for the current environment's database, exactly like the production
templates in `semantic-views/definitions/`. We then create the view with the Snowflake primitive
`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML` — the same call the deploy tooling makes under the hood.

In [ ]:
from agent_management.render_template import render_string

# Author the semantic view. {{ env.database }} is templated; the sandbox schema
# is injected directly so the demo stays self-contained.
SV_NAME = "SEM_DEMO_INCIDENTS"

sv_yaml_template = f"""
name: {SV_NAME.lower()}
description: Daily ski-resort safety incident summary (quickstart demo)
tables:
  - name: DEMO_DAILY_INCIDENTS
    synonyms:
      - incidents
      - safety
    description: One row per day with incident counts and response metrics
    base_table:
      database: {{{{ env.database }}}}
      schema: {SANDBOX_SCHEMA}
      table: DEMO_DAILY_INCIDENTS
    primary_key:
      columns:
        - INCIDENT_DATE
    dimensions:
      - name: INCIDENT_DATE
        description: Calendar date of the incidents
        expr: INCIDENT_DATE
        data_type: DATE
      - name: SKI_SEASON
        description: Ski season identifier (e.g. 2024-2025)
        expr: SKI_SEASON
        data_type: VARCHAR(16777216)
    facts:
      - name: INCIDENT_COUNT
        description: Number of incidents that day
        expr: INCIDENT_COUNT
        data_type: "NUMBER(38,0)"
        access_modifier: public_access
      - name: AVG_RESPONSE_MINUTES
        description: Average ski-patrol response time that day
        expr: AVG_RESPONSE_MINUTES
        data_type: "NUMBER(38,1)"
        access_modifier: public_access
    metrics:
      - name: TOTAL_INCIDENTS
        description: Total incidents across the selected period
        expr: SUM(DEMO_DAILY_INCIDENTS.INCIDENT_COUNT)
        access_modifier: public_access
      - name: AVG_DAILY_RESPONSE
        description: Average daily patrol response time
        expr: AVG(DEMO_DAILY_INCIDENTS.AVG_RESPONSE_MINUTES)
        access_modifier: public_access
"""

# Resolve {{ env.* }} placeholders for the current environment.
sv_yaml = render_string(sv_yaml_template, env_cfg)
print(sv_yaml)

In [ ]:
# Create the semantic view from the authored YAML. This is the exact primitive
# the YAML deploy path uses (see agent_management/semantic_views/deploy_yaml.py).
if RUN_LIVE:
    cur = conn.cursor()
    try:
        cur.execute(f"CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML('{SANDBOX_FQN}', $${sv_yaml}$$)")
        print(cur.fetchone()[0])
    finally:
        cur.close()
else:
    print(f"[skipped] would CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML('{SANDBOX_FQN}', ...)")

# Verify it exists.
rows = run_sql(f"SHOW SEMANTIC VIEWS LIKE '{SV_NAME}' IN SCHEMA {SANDBOX_FQN}", fetch=True)
if rows:
    print(f"\nCreated semantic view: {SANDBOX_FQN}.{SV_NAME}")

## Step 3 — Author and deploy an agent

A Cortex Agent wraps one or more tools with routing instructions. Our demo agent has a single
`cortex_analyst_text_to_sql` tool pointing at the semantic view we just created.

Key parts of the spec:
- **metadata.name** — short name; the env `name_suffix` is appended on deploy (`..._DEV`).
- **tools** — the Analyst tool with `semantic_view` set to our sandbox SV FQN.
- **instructions.orchestration / response** — how the agent routes and formats answers.

We deploy with the library's `deploy_agent(spec_path=...)`, which runs the full Cortex Agent Versioning
lifecycle (ADD LIVE VERSION → MODIFY SPEC → COMMIT → move alias). We write the spec to a temp file so
the demo never pollutes the tracked `agents/specs/` directory — the final section shows where it goes
for real.

In [ ]:
import tempfile

import yaml

from agent_management.agents.deploy import resolve_agent_identity
from agent_management.render_template import render_file

AGENT_SHORT_NAME = "incident_summary_demo"

# The tool points at the sandbox SV. {{ env.database }} is templated on render.
agent_spec_yaml = f"""
metadata:
  name: {AGENT_SHORT_NAME}
  version: "0.1.0"
  owner: quickstart_demo
  status: active

profile:
  display_name: Incident Summary Demo
  color: blue

description: >
  Demo agent that answers questions about daily ski-resort safety incidents
  using the SEM_DEMO_INCIDENTS semantic view.

sample_questions:
  - How many incidents happened on the busiest day?
  - What was the average patrol response time this season?

tools:
  - name: IncidentSummaryAnalytics
    type: cortex_analyst_text_to_sql
    semantic_view: {{{{ env.database }}}}.{SANDBOX_SCHEMA}.{SV_NAME}
    warehouse: {{{{ env.warehouse }}}}
    description: >
      PURPOSE: Daily safety incident counts and patrol response times.
      KEY METRICS: TOTAL_INCIDENTS, AVG_DAILY_RESPONSE.
      KEY DIMENSIONS: INCIDENT_DATE, SKI_SEASON.
      USE FOR: incident trends, busiest days, response-time questions.

instructions:
  response: >
    Be concise. Lead with the number. Format response times in minutes.
  orchestration: >
    You answer questions about daily safety incidents using IncidentSummaryAnalytics.
    "This season" = the most recent SKI_SEASON present in the data.
"""

# Write to a temp spec file and render {{ env.* }} placeholders.
tmp_dir = Path(tempfile.mkdtemp())
spec_path = tmp_dir / f"{AGENT_SHORT_NAME}.yml"
spec_path.write_text(agent_spec_yaml)

rendered = render_file(str(spec_path), env_cfg)
agent_dict = yaml.safe_load(rendered)
AGENT_NAME, _, AGENT_FQN = resolve_agent_identity(agent_dict, env_cfg)
print("Agent will deploy as:", AGENT_FQN)
print(rendered)

In [ ]:
from agent_management.agents.deploy import deploy_agent

# Deploy via the versioning path. dry_run mirrors RUN_LIVE so a read-through
# never mutates Snowflake. Reuse our connection so the agent is created under
# the same (privileged) role/session as the rest of the demo.
result = deploy_agent(
    agent_fqn=AGENT_FQN,
    spec_path=spec_path,
    env=ENV,
    connection=conn,
    dry_run=not RUN_LIVE,
)
print(result)

## Step 4 — Prove it works (smoke test)

Before writing formal evaluations, confirm the agent actually answers a question end to end. The
`agent-mgmt-smoke-agent` CLI sends a real prompt through the deployed agent and checks it responds
within a latency ceiling — the same smoke test the deploy workflows run.

In [ ]:
from agent_management.agents.smoke import run_smoke_test

DEPLOY_ALIAS = env_cfg.get("agent", {}).get("deploy_alias", "latest")

# Send a real prompt through the deployed agent over the Cortex REST API, reusing
# our connection/session so it runs under the same role that created the agent.
if RUN_LIVE:
    smoke = run_smoke_test(
        AGENT_FQN,
        env=ENV,
        alias=DEPLOY_ALIAS,
        prompts=["How many incidents happened on the busiest day?"],
        connection=conn,
    )
    print(smoke)
else:
    print(f"[skipped] smoke test for {AGENT_FQN} (alias={DEPLOY_ALIAS}) — set RUN_LIVE=True")

## Step 5 — Versioning and promotion

Every deploy commits a new immutable `VERSION$N`. **Aliases** are moving pointers to a version, and
that's how you promote a tested build forward without redeploying:

- **dev** manages `latest` — every dev deploy moves it.
- **prod** manages `validated` (passed gates, pre-customer) and `production` (live customer traffic).
- a `DEFAULT` alias backs selectorless REST calls.

Promotion is just "point the next alias at the version the previous alias already trusts" — no rebuild.
That's exactly what `promote-validated-to-production.yml` does via
`agent-mgmt-agent-versioning promote`.

Below we demonstrate the full mechanic live on the demo agent: inspect versions, ship a change as a new
version, then move `latest -> validated -> production`. (In real life dev only manages `latest`; we show
the prod aliases here on the throwaway agent purely to illustrate the movement.)

In [ ]:
from agent_management.agents import get_aliases, list_versions, promote_alias, set_alias


def show_versions(label: str):
    print(f"--- {label} ---")
    for v in list_versions(conn, AGENT_FQN, include_live=True):
        flags = " [DEFAULT]" if v.is_default else ""
        print(f"  {v.name:10} alias={v.alias or '-':10} created={v.created}{flags}")
    print("  aliases:", get_aliases(conn, AGENT_FQN))


if RUN_LIVE:
    show_versions("after first deploy")
else:
    print("[skipped] version inspection — set RUN_LIVE=True")

### Ship a change as a new version

Edit the spec and redeploy. `deploy_agent` adds a new live version from the last one, applies the change,
commits it as a fresh `VERSION$N`, and moves the `latest` alias to it — the previous version stays intact
for rollback.

In [ ]:
# Make a small change (bump version, add a sample question) and redeploy.
agent_spec_yaml_v2 = agent_spec_yaml.replace(
    'version: "0.1.0"', 'version: "0.2.0"'
).replace(
    "  - What was the average patrol response time this season?",
    "  - What was the average patrol response time this season?\n"
    "  - Which incident day had the slowest patrol response?",
)

spec_path_v2 = tmp_dir / f"{AGENT_SHORT_NAME}_v2.yml"
spec_path_v2.write_text(agent_spec_yaml_v2)

result_v2 = deploy_agent(
    agent_fqn=AGENT_FQN,
    spec_path=spec_path_v2,
    env=ENV,
    connection=conn,
    dry_run=not RUN_LIVE,
)
print(result_v2)

if RUN_LIVE:
    show_versions("after second deploy (latest should have moved)")

### Promote across environments (the alias mechanic)

To promote, pin the trusted version to the next alias. We pin the current `latest` version to
`validated`, then promote `validated -> production`. `promote_alias` moves the `to_alias` to whatever
version the `from_alias` already points at — no rebuild, fully auditable.

CLI equivalents (what CI runs):

```bash
agent-mgmt-agent-versioning list --env dev --agent INCIDENT_SUMMARY_DEMO
agent-mgmt-agent-versioning promote --env prod --from validated --to production
```

Note: `get_aliases` reports one alias per version, so when several aliases point at the same version you
may see only the most recently set one in the dict — the others are still valid pointers to that version.

In [ ]:
if RUN_LIVE:
    # get_aliases returns uppercase keys (LATEST, DEFAULT, ...); normalize lookup.
    aliases = {k.lower(): v for k, v in get_aliases(conn, AGENT_FQN).items()}
    latest_version = aliases["latest"]

    # 1. Pin the version currently behind `latest` to `validated` (passed gates).
    set_alias(conn, AGENT_FQN, version=latest_version, alias="validated")
    print("after set validated:", get_aliases(conn, AGENT_FQN))

    # 2. Promote validated -> production (move `production` to validated's version).
    moved_to = promote_alias(conn, AGENT_FQN, from_alias="validated", to_alias="production")
    print(f"promoted production -> {moved_to}")
    print("after promote:", get_aliases(conn, AGENT_FQN))
else:
    print("[skipped] alias promotion — set RUN_LIVE=True")

### Rollback is the same mechanic, in reverse

Because versions are immutable and aliases move, rollback is just pointing an alias back at a known-good
version — no redeploy:

```python
set_alias(conn, AGENT_FQN, version="VERSION$1", alias="production")
```

In CI this is `agent-mgmt-rollback --env prod --agent <name> --alias production`, which
`rollback.yml` runs and then re-smoke-tests.

## Step 6 — Create a golden dataset and eval

A smoke test proves the agent responds. **Evaluations** prove it responds *correctly*, repeatably.

This framework uses **dynamic ground truth**: each golden question carries a `validation_query` (SQL
run at eval time against the source data) plus an `answer_template` that formats the query result into
the expected natural-language answer. This keeps golden answers correct even as data changes.

We author two files:
- a **golden dataset** (`questions[]` with `validation_query` + `answer_template`), and
- an **eval config** (which agent, which dataset, thresholds, metrics).

Then `agent-mgmt-eval-agent` renders these, runs the agent over each question, and scores
`answer_correctness` and `logical_consistency` against the thresholds in the env config.

In [ ]:
# Author the golden dataset. Validation queries hit the sandbox demo table via
# the {{ env.database }} placeholder so they render correctly in any environment.
golden_dataset_yaml = f"""
# Golden questions for INCIDENT_SUMMARY_DEMO (dynamic ground truth)
questions:
  - question: "How many total incidents occurred this season?"
    expected_tools: ["IncidentSummaryAnalytics"]
    category: incidents
    test_type: in_scope
    validation_query: |
      WITH current_season AS (
        SELECT MAX(SKI_SEASON) AS season
        FROM {{{{ env.database }}}}.{SANDBOX_SCHEMA}.DEMO_DAILY_INCIDENTS
      )
      SELECT SKI_SEASON AS season,
             SUM(INCIDENT_COUNT) AS total_incidents
      FROM {{{{ env.database }}}}.{SANDBOX_SCHEMA}.DEMO_DAILY_INCIDENTS
      WHERE SKI_SEASON = (SELECT season FROM current_season)
      GROUP BY SKI_SEASON
    answer_template: >-
      The {{season}} season had {{total_incidents:,}} total incidents.

  - question: "Which day had the most incidents this season?"
    expected_tools: ["IncidentSummaryAnalytics"]
    category: incidents
    test_type: in_scope
    validation_query: |
      WITH current_season AS (
        SELECT MAX(SKI_SEASON) AS season
        FROM {{{{ env.database }}}}.{SANDBOX_SCHEMA}.DEMO_DAILY_INCIDENTS
      )
      SELECT INCIDENT_DATE, INCIDENT_COUNT
      FROM {{{{ env.database }}}}.{SANDBOX_SCHEMA}.DEMO_DAILY_INCIDENTS
      WHERE SKI_SEASON = (SELECT season FROM current_season)
      ORDER BY INCIDENT_COUNT DESC
      LIMIT 1
    answer_template: >-
      The busiest day was {{INCIDENT_DATE}} with {{INCIDENT_COUNT}} incidents.

  - question: "What was the average patrol response time this season?"
    expected_tools: ["IncidentSummaryAnalytics"]
    category: incidents
    test_type: in_scope
    validation_query: |
      WITH current_season AS (
        SELECT MAX(SKI_SEASON) AS season
        FROM {{{{ env.database }}}}.{SANDBOX_SCHEMA}.DEMO_DAILY_INCIDENTS
      )
      SELECT ROUND(AVG(AVG_RESPONSE_MINUTES), 1) AS avg_response
      FROM {{{{ env.database }}}}.{SANDBOX_SCHEMA}.DEMO_DAILY_INCIDENTS
      WHERE SKI_SEASON = (SELECT season FROM current_season)
    answer_template: >-
      Average patrol response time this season was {{avg_response}} minutes.
"""

dataset_path = tmp_dir / f"{AGENT_SHORT_NAME}_eval.yaml"
dataset_path.write_text(render_string(golden_dataset_yaml, env_cfg))
print(dataset_path.read_text())

In [ ]:
# Author the eval config: which agent, which dataset, thresholds, metrics.
# Note: agent.name uses the UNSUFFIXED base name and lets eval.source_database /
# eval.agents_schema resolve the environment, matching the repo's existing
# configs (e.g. agent-evaluation/configs/ski_ops_assistant.yaml).
AGENT_BASE_NAME = AGENT_SHORT_NAME.upper()

eval_config_yaml = f"""
agent:
  name: "{AGENT_BASE_NAME}"
  database: "{{{{ eval.source_database }}}}"
  schema: "{{{{ eval.agents_schema }}}}"

dataset:
  questions: "datasets/{AGENT_SHORT_NAME}_eval.yaml"
  snowflake_table: "{{{{ eval.source_database }}}}.{{{{ eval.agents_schema }}}}.{AGENT_BASE_NAME}_EVAL_DATA"

snowflake:
  stage: "{{{{ eval.stage }}}}"
  file_format: "{{{{ eval.file_format }}}}"
  warehouse: "{{{{ eval.warehouse }}}}"

evaluation:
  label: "{AGENT_BASE_NAME} evaluation"
  description: "Answer correctness + logical consistency for the incident summary demo"

thresholds:
  answer_correctness: {{{{ eval.thresholds.answer_correctness }}}}
  logical_consistency: {{{{ eval.thresholds.logical_consistency }}}}

metrics:
  - "answer_correctness"
  - "logical_consistency"
"""

eval_config_path = tmp_dir / f"{AGENT_SHORT_NAME}.yaml"
eval_config_path.write_text(render_string(eval_config_yaml, env_cfg))
print(eval_config_path.read_text())

### See dynamic ground truth in action

Before running the full eval harness, here is the core mechanic live: execute one question's
`validation_query` and format the result with its `answer_template`. This is the *expected* answer the
agent's response is scored against.

The full evaluation (`agent-mgmt-eval-agent`) does this for every question, sends each question through
the deployed agent, and scores the agent's answer with an LLM judge against the thresholds. It discovers
configs from `agent-evaluation/configs/`, so to run it for this demo you place the two files there
(shown in the next section) and run:

```bash
agent-mgmt-eval-agent --env dev --agent INCIDENT_SUMMARY_DEMO
```

In [ ]:
# Live demonstration of the dynamic ground-truth mechanic for one question.
demo_question = yaml.safe_load(dataset_path.read_text())["questions"][0]
print("Q:", demo_question["question"])

if RUN_LIVE:
    cur = conn.cursor()
    try:
        cur.execute(demo_question["validation_query"])
        cols = [c[0].lower() for c in cur.description]
        row = dict(zip(cols, cur.fetchone()))
    finally:
        cur.close()
    expected = demo_question["answer_template"].format(**row)
    print("Ground-truth row:", row)
    print("Expected answer :", expected.strip())
else:
    print("[skipped] would run validation_query and format the answer_template")

## Step 7 — Promote into the repo

Everything above was authored in temp files so the demo stays clean. To make these artifacts part of the
framework (validated in CI, deployed by the pipeline), copy them to their home locations:

| Artifact (this notebook) | Repo location | Consumed by |
|---|---|---|
| Semantic view YAML (`sv_yaml_template`) | `semantic-views/definitions/sem_demo_incidents.yaml` | `validate-pr.yml`, deploy workflows |
| Agent spec (`agent_spec_yaml`) | `agents/specs/incident_summary_demo.yml` | `agent-mgmt-deploy-agents` |
| Register the agent | add under `agents:` in `project.yml` | SV-eval scoping, deploy discovery |
| Eval config (`eval_config_yaml`) | `agent-evaluation/configs/incident_summary_demo.yaml` | `agent-mgmt-eval-agent` |
| Golden dataset (`golden_dataset_yaml`) | `agent-evaluation/datasets/incident_summary_demo_eval.yaml` | `agent-mgmt-eval-agent` |

Keep the `{{ env.* }}` / `{{ eval.* }}` placeholders in the files (do not hardcode the rendered values) —
the templating is what lets one definition deploy to both `dev` and `prod`.

Add this block to `project.yml` so the agent is registered:

```yaml
agents:
  incident_summary_demo:
    semantic_views:
      - SEM_DEMO_INCIDENTS
```

Then the normal flow takes over:

```bash
agent-mgmt-validate --env dev          # local lint + render
git add semantic-views/ agents/ agent-evaluation/ project.yml
git commit -m "Add incident summary demo agent"
# PR -> dev (deploy-dev.yml) -> PR -> main (deploy-prod-validated.yml)
```

```mermaid
flowchart LR
    feat[feature branch] -->|PR| dev[dev: deploy-dev]
    dev -->|PR| main[main: deploy-prod-validated]
    main -->|manual| prod[promote-validated-to-production]
```

## Teardown

Drop everything the demo created so it leaves no residue. Run this when you're done exploring (skip it
if you want to keep the demo objects around to inspect in Snowsight).

In [ ]:
# Drop the demo agent, semantic view, table, and sandbox schema.
if RUN_LIVE:
    run_sql(f"DROP AGENT IF EXISTS {AGENT_FQN}")
    run_sql(f"DROP SEMANTIC VIEW IF EXISTS {SANDBOX_FQN}.{SV_NAME}")
    run_sql(f"DROP TABLE IF EXISTS {SANDBOX_FQN}.DEMO_DAILY_INCIDENTS")
    run_sql(f"DROP SCHEMA IF EXISTS {SANDBOX_FQN}")
    conn.close()
    print("\nTeardown complete — demo objects removed.")
else:
    print("[skipped] teardown (RUN_LIVE=False — nothing was created)")